# Guarded HelpSteer3 PPO pipeline

This GPU-only notebook runs the scale-aware, domain-balanced PPO experiment in restartable segments. Training uses the frozen SFT policy and two-epoch reward model, a 768-token rollout budget, batch size 64, KL coefficient 0.10, reward whitening disabled, exact EOS replacement, quantile reward bounds, and smooth repetition shaping.

Every target cell resumes from the latest exact checkpoint and stops at a cumulative update. Checkpoints are saved and copied to Drive every 25 updates. After a runtime reset, rerun Sections 0-2, then jump directly to the next unfinished target. Evaluation gates are informative; they never block a later training cell.

## 0. Runtime, Drive, and repository

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/djdhillxn/rlhf.git"
REPO_ROOT = Path("/content/rlhf")
LOCAL_RUN_ROOT = Path("/content/rlhf_runs/qwen25_05b_helpsteer3_trl_a100_full/full")
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/8mile/rlhf_runs/qwen25_05b_helpsteer3_trl_a100_full/full")
DRIVE_LIGHTWEIGHT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/8mile/rlhf_runs_lightweight_export/qwen25_05b_helpsteer3_trl_a100_full/full")

PPO_RUN_NAME = "ppo_guarded_r768_b64_kl10_rm_ep2"
PPO_CONFIG_SOURCE = REPO_ROOT / "configs/trl/qwen25_05b_helpsteer3_ppo.yaml"
EVAL_CONFIG_SOURCE = REPO_ROOT / "configs/trl/qwen25_05b_helpsteer3_eval_suite.yaml"
RUNTIME_CONFIG = Path("/content/ppo_guarded_runtime.yaml")

CACHE_DIR = LOCAL_RUN_ROOT / "data"
SFT_MODEL = LOCAL_RUN_ROOT / "sft" / "final_merged_model"
REWARD_DIR = LOCAL_RUN_ROOT / "reward_epoch2"
REWARD_MODEL = REWARD_DIR / "final_merged_model"
REWARD_CENTER = REWARD_DIR / "reward_center.json"
PPO_DIR = LOCAL_RUN_ROOT / PPO_RUN_NAME
DRIVE_PPO_DIR = DRIVE_RUN_ROOT / PPO_RUN_NAME

for path in (LOCAL_RUN_ROOT, CACHE_DIR, REWARD_DIR, PPO_DIR, DRIVE_PPO_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Local run root:", LOCAL_RUN_ROOT)
print("Drive run root:", DRIVE_RUN_ROOT)
print("Guarded PPO run:", PPO_RUN_NAME)

In [ ]:
if not (REPO_ROOT / ".git").is_dir():
    !git clone {REPO_URL} {REPO_ROOT}

%cd {REPO_ROOT}
!git pull --ff-only
!python -m pip install -q -e ".[trl]"

%env PYTHONUNBUFFERED=1
%env TOKENIZERS_PARALLELISM=false
%env TRL_EXPERIMENTAL_SILENCE=1

In [ ]:
import torch
import transformers
import trl

assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
assert torch.cuda.device_count() == 1, "This exact balanced-resume run expects one GPU process."
properties = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory: {properties.total_memory / 2**30:.1f} GiB")
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)

## 1. Restore immutable inputs and interrupted state

The SFT policy, epoch-two reward model, reward center, prepared PPO/reward datasets, and any prior guarded PPO checkpoints are copied from Drive to the faster Colab SSD. Missing guarded-PPO output is normal on the first launch.

In [ ]:
!python -m scripts.rlhf_sync_runs \
    --source "{DRIVE_RUN_ROOT / 'sft' / 'final_merged_model'}" \
    --destination "{SFT_MODEL}" \
    --profile full

!python -m scripts.rlhf_sync_runs \
    --source "{DRIVE_RUN_ROOT / 'reward_epoch2'}" \
    --destination "{REWARD_DIR}" \
    --profile full \
    --include final_merged_model \
    --include reward_center.json \
    --include reward_audit.json

!python -m scripts.rlhf_sync_runs \
    --source "{DRIVE_RUN_ROOT / 'data'}" \
    --destination "{CACHE_DIR}" \
    --profile full \
    --include ppo \
    --include reward \
    --include preparation_report.json \
    --include tokenizer

!python -m scripts.rlhf_sync_runs \
    --source "{DRIVE_PPO_DIR}" \
    --destination "{PPO_DIR}" \
    --profile full \
    --allow-missing

print("SFT ready:", (SFT_MODEL / "config.json").is_file())
print("Reward model ready:", (REWARD_MODEL / "config.json").is_file())
print("Reward center ready:", REWARD_CENTER.is_file())
print("PPO data ready:", (CACHE_DIR / "ppo" / "train" / "dataset_info.json").is_file())
print("Reward data ready:", (CACHE_DIR / "reward" / "train" / "dataset_info.json").is_file())
print("Restored checkpoints:", sorted(path.name for path in PPO_DIR.glob("checkpoint-*")))

## 2. One visible runtime configuration

This is the only static configuration cell. Later training cells change only `train.target_update`, the cumulative stopping point. Resume validation fingerprints every optimization, model, LoRA, reward-guardrail, and rollout-balance setting while intentionally allowing the target and storage paths to vary.

In [ ]:
import yaml

runtime_cfg = yaml.safe_load(PPO_CONFIG_SOURCE.read_text(encoding="utf-8"))
runtime_cfg["model"].update({
    "policy_model_path": str(SFT_MODEL),
    "reference_model_path": str(SFT_MODEL),
    "reward_model_path": str(REWARD_MODEL),
    "reward_center_path": str(REWARD_CENTER),
    "value_model_path": str(REWARD_MODEL),
})
runtime_cfg["data"]["cache_dir"] = str(CACHE_DIR)
runtime_cfg["train"].update({
    "output_dir": str(PPO_DIR),
    "checkpoint_sync_dir": str(DRIVE_PPO_DIR),
    "final_sync_dir": str(DRIVE_PPO_DIR),
    "resume_from_checkpoint": "auto",
    "target_update": 25,
})
RUNTIME_CONFIG.write_text(
    yaml.safe_dump(runtime_cfg, sort_keys=False), encoding="utf-8"
)
print(RUNTIME_CONFIG.read_text(encoding="utf-8"))

In [ ]:
import json

audit_path = REWARD_DIR / "reward_audit.json"
if audit_path.is_file():
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    print("Reward-model validation accuracy:", audit.get("accuracy"))
    print("Reward-model domain accuracy:", json.dumps(audit.get("by_domain", {}), indent=2))
else:
    print("Reward audit was not restored; training can proceed, but retain it for provenance.")

## 3. Preflight for the first segment

The doctor checks CUDA, tokenizer semantics, prepared data, writable storage, guarded-PPO invariants, and exact-resume health when a checkpoint exists. On later sessions, run the doctor command inside the target cell you are about to execute.

In [ ]:
!python -m scripts.rlhf_trl_doctor \
    --config {RUNTIME_CONFIG} \
    --stage ppo \
    --set train.target_update=25
!python -m scripts.rlhf_trl_train_ppo \
    --config {RUNTIME_CONFIG} \
    --prepare-guardrails-only

guardrails = json.loads((PPO_DIR / "ppo_reward_guardrails.json").read_text(encoding="utf-8"))
print(json.dumps(guardrails, indent=2))

## 4. Segmented PPO training

Run target cells in order. A target is cumulative: update 50 restores update 25 and performs 25 new updates. Do not rerun a target already reached. Each process writes live TQDM progress, one diagnostic JSONL row per update, an exact-resume checkpoint, a checkpoint health report, and a merged policy snapshot. It then synchronizes the complete run directory to Drive.

### Target update 25

In [ ]:
!python -m scripts.rlhf_trl_doctor --config {RUNTIME_CONFIG} --stage ppo --set train.target_update=25
!python -m scripts.rlhf_trl_train_ppo --config {RUNTIME_CONFIG} --set train.target_update=25

### Target update 50

In [ ]:
!python -m scripts.rlhf_trl_doctor --config {RUNTIME_CONFIG} --stage ppo --set train.target_update=50
!python -m scripts.rlhf_trl_train_ppo --config {RUNTIME_CONFIG} --set train.target_update=50

### Target update 75

In [ ]:
!python -m scripts.rlhf_trl_doctor --config {RUNTIME_CONFIG} --stage ppo --set train.target_update=75
!python -m scripts.rlhf_trl_train_ppo --config {RUNTIME_CONFIG} --set train.target_update=75

### Target update 100

In [ ]:
!python -m scripts.rlhf_trl_doctor --config {RUNTIME_CONFIG} --stage ppo --set train.target_update=100
!python -m scripts.rlhf_trl_train_ppo --config {RUNTIME_CONFIG} --set train.target_update=100

### Target update 125

In [ ]:
!python -m scripts.rlhf_trl_doctor --config {RUNTIME_CONFIG} --stage ppo --set train.target_update=125
!python -m scripts.rlhf_trl_train_ppo --config {RUNTIME_CONFIG} --set train.target_update=125

### Target update 150

In [ ]:
!python -m scripts.rlhf_trl_doctor --config {RUNTIME_CONFIG} --stage ppo --set train.target_update=150
!python -m scripts.rlhf_trl_train_ppo --config {RUNTIME_CONFIG} --set train.target_update=150

### Target update 175

In [ ]:
!python -m scripts.rlhf_trl_doctor --config {RUNTIME_CONFIG} --stage ppo --set train.target_update=175
!python -m scripts.rlhf_trl_train_ppo --config {RUNTIME_CONFIG} --set train.target_update=175

### Target update 188

In [ ]:
!python -m scripts.rlhf_trl_doctor --config {RUNTIME_CONFIG} --stage ppo --set train.target_update=188
!python -m scripts.rlhf_trl_train_ppo --config {RUNTIME_CONFIG} --set train.target_update=188

## 5. Checkpoint health and live diagnostics

Run this after any segment. The report is descriptive, not a gate: it surfaces response-level and per-token KL, raw and shaped reward percentiles, clipping, EOS/cap rates, response lengths, repetition, and all four domain slices.

In [ ]:
def show_update(update):
    health_path = PPO_DIR / f"checkpoint-{update}-health.json"
    diagnostics_path = PPO_DIR / "ppo_diagnostics.jsonl"
    if not health_path.is_file():
        print(f"No health report for update {update}")
        return
    health = json.loads(health_path.read_text(encoding="utf-8"))
    print("Checkpoint healthy:", health["healthy"])
    rows = []
    if diagnostics_path.is_file():
        rows = [json.loads(line) for line in diagnostics_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    matching = [row for row in rows if int(row.get("update", -1)) == int(update)]
    if not matching:
        print("No diagnostic row found for this update.")
        return
    row = matching[-1]
    keys = [
        "episode", "objective/kl", "objective/kl_per_response_token",
        "objective/rlhf_reward", "objective/scores",
        "guardrail/rm_score_p05", "guardrail/rm_score_p50", "guardrail/rm_score_p95",
        "guardrail/reward_clipped_fraction", "guardrail/repetition_penalty_mean",
        "rollout/eos_rate", "rollout/cap_rate", "rollout/response_length_mean",
        "rollout/repeated_token_4gram_fraction_mean", "rollout/repeated_token_4gram_fraction_p95",
    ]
    for key in keys:
        print(f"{key}: {row.get(key)}")
    for domain in ("code", "general", "stem", "multilingual"):
        print(domain, {
            "score": row.get(f"domain/{domain}/shaped_score_mean"),
            "eos": row.get(f"domain/{domain}/eos_rate"),
            "length": row.get(f"domain/{domain}/response_length_mean"),
            "repeat4": row.get(f"domain/{domain}/repeated_token_4gram_fraction_mean"),
        })

show_update(25)  # Change only this number after a later segment.

## 6. Evaluation configuration helper

The 256-prompt gate selects exactly 64 prompts from each domain. The full gate uses all 2,017 validation prompts. Evaluate immediately after the corresponding training target while `final_merged_policy` represents that target.

In [ ]:
def write_eval_config(update, full=False):
    selection_path = PPO_DIR / "final_merged_policy" / "ppo_checkpoint_selection.json"
    if not selection_path.is_file():
        raise FileNotFoundError("The latest merged PPO snapshot has no checkpoint provenance.")
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    if int(selection.get("selected_update", -1)) != int(update):
        raise ValueError(f"final_merged_policy is update {selection.get('selected_update')}, not requested update {update}.")
    cfg = yaml.safe_load(EVAL_CONFIG_SOURCE.read_text(encoding="utf-8"))
    suffix = "full2017" if full else "stratified256"
    output_dir = LOCAL_RUN_ROOT / f"eval_{PPO_RUN_NAME}_u{update}_{suffix}"
    cfg["experiment"]["id"] = f"{PPO_RUN_NAME}-u{update}-{suffix}"
    cfg["model"].update({
        "name": "Qwen/Qwen2.5-0.5B-Instruct",
        "tokenizer_path": str(SFT_MODEL),
        "local_files_only": False,
    })
    cfg["reward_model"].update({
        "checkpoint_dir": str(REWARD_MODEL),
        "reward_center_path": str(REWARD_CENTER),
    })
    cfg["policies"][0]["checkpoint_dir"] = None
    cfg["policies"][1]["checkpoint_dir"] = str(SFT_MODEL)
    cfg["policies"][2]["checkpoint_dir"] = str(PPO_DIR / "final_merged_policy")
    cfg["generation"].update({
        "max_prompt_length": 3072,
        "max_new_tokens": 768,
    })
    cfg["eval"].update({
        "output_dir": str(output_dir),
        "num_prompts": "all" if full else 256,
        "shuffle": False,
        "stratify_by_domain": not full,
        "batch_size": 16,
        "resume": True,
    })
    path = PPO_DIR / f"eval_u{update}_{suffix}.yaml"
    path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    return path, output_dir

print("Evaluation helper ready.")

## 7. Stratified 256-prompt gates

Run the matching cell after updates 25, 50, 75, 100, 125, 150, and 188. Change `UPDATE` only to one of those values. The qualitative audit reuses the same deterministic repetition and reward-mismatch diagnostics as the final suite.

In [ ]:
UPDATE = 25  # Use 25, 50, 75, 100, 125, 150, or 188.
EVAL_CONFIG, EVAL_DIR = write_eval_config(UPDATE, full=False)

!python -m scripts.rlhf_evaluate_policy_suite --config {EVAL_CONFIG}
!python -m scripts.rlhf_audit_policy_suite \
    --eval-dir "{EVAL_DIR}" \
    --base-label base \
    --sft-label sft_trl \
    --ppo-label ppo_trl
!python -m scripts.rlhf_sync_runs \
    --source "{EVAL_DIR}" \
    --destination "{DRIVE_RUN_ROOT / EVAL_DIR.name}" \
    --profile full

## 8. Full 2,017-prompt gates

Run only at updates 100, 150, and 188. These are the expensive checkpoint-selection evaluations.

In [ ]:
UPDATE = 100  # Use 100, 150, or 188.
FULL_CONFIG, FULL_EVAL_DIR = write_eval_config(UPDATE, full=True)

!python -m scripts.rlhf_evaluate_policy_suite --config {FULL_CONFIG}
!python -m scripts.rlhf_audit_policy_suite \
    --eval-dir "{FULL_EVAL_DIR}" \
    --base-label base \
    --sft-label sft_trl \
    --ppo-label ppo_trl
!python -m scripts.rlhf_sync_runs \
    --source "{FULL_EVAL_DIR}" \
    --destination "{DRIVE_RUN_ROOT / FULL_EVAL_DIR.name}" \
    --profile full

## 9. Compare gates and select a checkpoint

Selection is deliberately manual. Review reward preference, EOS/cap completion, repetition, KL per generated token, all domain slices, and the qualitative audit together. The learned reward model is not an independent judge, so do not select a checkpoint from its score alone.

In [ ]:
import pandas as pd

rows = []
for update in (25, 50, 75, 100, 125, 150, 188):
    eval_dir = LOCAL_RUN_ROOT / f"eval_{PPO_RUN_NAME}_u{update}_stratified256"
    summary_path = eval_dir / "policy_suite_summary.json"
    if not summary_path.is_file():
        continue
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    pairwise = summary.get("pairwise", {}).get("base_vs_ppo_trl", {})
    domain_counts = pairwise.get("domain_winner_counts", {})
    diagnostics = [
        json.loads(line)
        for line in (PPO_DIR / "ppo_diagnostics.jsonl").read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    diagnostic = next((row for row in reversed(diagnostics) if int(row.get("update", -1)) == update), {})
    audit_csv = eval_dir / "curation_ppo_repetition_risks.csv"
    repetition_risks = max(0, len(audit_csv.read_text(encoding="utf-8").splitlines()) - 1) if audit_csv.is_file() else None
    row = {
        "update": update,
        "ppo_win_rate_vs_base": pairwise.get("ppo_trl_win_rate"),
        "mean_reward_delta_vs_base": pairwise.get("ppo_trl_minus_base", {}).get("mean"),
        "eval_cap_rate": summary.get("per_policy", {}).get("ppo_trl", {}).get("cap_hit_rate"),
        "eval_repetition_risks": repetition_risks,
        "kl_per_token": diagnostic.get("objective/kl_per_response_token"),
        "eos_rate": diagnostic.get("rollout/eos_rate"),
        "repeat4_p95": diagnostic.get("rollout/repeated_token_4gram_fraction_p95"),
        "clipped_fraction": diagnostic.get("guardrail/reward_clipped_fraction"),
    }
    for domain in ("code", "general", "stem", "multilingual"):
        counts = domain_counts.get(domain, {})
        wins = int(counts.get("ppo_trl", 0))
        losses = int(counts.get("base", 0))
        row[f"{domain}_ppo_win_rate"] = wins / max(wins + losses, 1)
    rows.append(row)

display(pd.DataFrame(rows))

In [ ]:
BEST_UPDATE = None  # Set to 100, 150, or 188 after reviewing the checkpoint gates.
assert BEST_UPDATE in {100, 150, 188}, "Choose an evaluated full-suite checkpoint first."
SELECTED_POLICY_DIR = PPO_DIR / f"selected_policy_u{BEST_UPDATE}"

!python -m scripts.rlhf_trl_train_ppo \
    --config {RUNTIME_CONFIG} \
    --export-checkpoint "{PPO_DIR / f'checkpoint-{BEST_UPDATE}'}" \
    --export-output-dir "{SELECTED_POLICY_DIR}"
!python -m scripts.rlhf_sync_runs \
    --source "{SELECTED_POLICY_DIR}" \
    --destination "{DRIVE_PPO_DIR / SELECTED_POLICY_DIR.name}" \
    --profile full

## 10. Lightweight artifact export

This Drive-to-Drive export omits model tensors while retaining configs, Trainer state, diagnostics, health reports, evaluation samples, CSV/JSON/JSONL, markdown audits, and plots. Google Drive for desktop can then make the analysis artifacts available to the local repository.

In [ ]:
!python -m scripts.rlhf_sync_runs \
    --source "{DRIVE_RUN_ROOT}" \
    --destination "{DRIVE_LIGHTWEIGHT_ROOT}" \
    --profile lightweight \
    --max-size-mb 50

print("Guarded PPO artifacts exported to:", DRIVE_LIGHTWEIGHT_ROOT)